<div style="color:#3c4d5a; border-top: 7px solid #42A5F5; border-bottom: 7px solid #42A5F5; padding: 5px; text-align: center; text-transform: uppercase"><h1>03. Predicción y Pruebas de Inferencia</h1> </div>

Desarrollado por: Alexis Guamán y Cinthya Ramón

El presente notebook tiene como objetivo demostrar el proceso de carga e inferencia de los modelos congelados de la Fase 1 (Phase 1 Final Release), verificando su integridad criptográfica antes de utilizarlos.

Este procedimiento garantiza que los artefactos de Machine Learning no han sido alterados desde su empaquetado, y permite comprobar que las predicciones son deterministas y reproducibles sobre muestras de los datasets de prueba.

La metodología se estructura en las siguientes etapas:

1. **Verificación de seguridad y manifiesto:** validación de hashes SHA-256 de todos los archivos del release.
2. **Carga de artefactos de ML:** importación de modelos y preprocesadores desde el directorio de liberación.
3. **Prueba de inferencia reactiva:** predicción del perfil de red sobre muestras del conjunto de test.
4. **Prueba de inferencia predictiva:** predicción de degradación con umbral personalizado sobre ventanas temporales de test.
5. **Conclusión:** verificación de la consistencia y determinismo del sistema.

**Contenido de este notebook:**

- [1. Verificación de Seguridad y Manifiesto](#1-verificacion-de-seguridad-y-manifiesto)

- [2. Carga de Artefactos de ML](#2-carga-de-artefactos-de-ml)

- [3. Prueba de Inferencia: Modelo Reactivo](#3-prueba-de-inferencia-modelo-reactivo)

- [4. Prueba de Inferencia: Modelo Predictivo](#4-prueba-de-inferencia-modelo-predictivo)

- [5. Conclusión](#5-conclusion)

In [ ]:
import pandas as pd
import numpy as np
import joblib
import json
import os
import hashlib


<div id="1-verificacion-de-seguridad-y-manifiesto" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>1. Verificación de Seguridad y Manifiesto</h2> </div>

Antes de cargar los pesos del modelo, se verifica el archivo `manifest.json` y la integridad criptográfica de cada artefacto mediante hashes SHA-256. Este paso es fundamental para detectar posibles alteraciones accidentales o maliciosas de los archivos del release (Data Poisoning), garantizando que los modelos utilizados corresponden exactamente a la versión congelada y auditada.

In [ ]:
RELEASE_DIR = 'models/phase1_final_release'
MANIFEST_PATH = os.path.join(RELEASE_DIR, 'manifest.json')

with open(MANIFEST_PATH, 'r') as f:
    manifest = json.load(f)

print(f"Versión de Liberación: {manifest['release_name']} ({manifest['release_version']})")

def sha256(filepath):
    h = hashlib.sha256()
    with open(filepath, 'rb') as file:
        while chunk := file.read(8192):
            h.update(chunk)
    return h.hexdigest()

print("\nVerificando hashes...")
for filename, expected_hash in manifest['sha256_hashes'].items():
    filepath = os.path.join(RELEASE_DIR, filename)
    actual_hash = sha256(filepath)
    if actual_hash == expected_hash:
        print(f"[OK] {filename}")
    else:
        print(f"[ERROR] {filename} HASH MISMATCH!")


<div id="2-carga-de-artefactos-de-ml" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>2. Carga de Artefactos de ML</h2> </div>

Se cargan los preprocesadores (`StandardScaler`) y los clasificadores (`DecisionTreeClassifier`, `RandomForestClassifier`) desde el directorio de liberación seguro (`models/phase1_final_release/`), utilizando las rutas definidas en el manifiesto para garantizar la trazabilidad de los artefactos.

In [ ]:
# Nombres definidos en el manifiesto
mod_r = joblib.load(os.path.join(RELEASE_DIR, manifest['reactive_model_path']))
mod_p = joblib.load(os.path.join(RELEASE_DIR, manifest['predictive_model_path']))
prep_r = joblib.load(os.path.join(RELEASE_DIR, manifest['preprocessors'][0]))
prep_p = joblib.load(os.path.join(RELEASE_DIR, manifest['preprocessors'][1]))

with open(os.path.join(RELEASE_DIR, manifest['artifacts'][0]), 'r') as f:
    meta = json.load(f)

print("Modelos cargados correctamente en memoria.")

<div id="3-prueba-de-inferencia-modelo-reactivo" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>3. Prueba de Inferencia: Modelo Reactivo</h2> </div>

Se selecciona una muestra aleatoria de 5 observaciones del conjunto de test del dataset reactivo. Se aplica el preprocesador (estandarización) y se obtienen las predicciones del modelo, comparándolas con las etiquetas reales para verificar el correcto funcionamiento del pipeline de inferencia.

In [ ]:
df_r = pd.read_csv('data/processed/dataset_reactivo.csv')
if 'ping_ms' in df_r.columns:
    df_r.rename(columns={'ping_ms': 'latency_ms'}, inplace=True)

sample_r = df_r[df_r['split'] == 'test'].sample(5, random_state=42)
X_r = sample_r[meta['reactive_features']]
y_true_r = sample_r['recommended_profile']

# Pipeline de inferencia
X_r_scaled = prep_r.transform(X_r)
preds_r = mod_r.predict(X_r_scaled)

print("--- INFERENCIA REACTIVA ---")
for i, (idx, row) in enumerate(X_r.iterrows()):
    print(f"Muestra {i+1} | Input: Upload={row['upload_mbps']:.2f}, Download={row['download_mbps']:.2f}, Latency={row['latency_ms']:.2f}")
    print(f"           | Predicción: {preds_r[i]} (Real: {y_true_r.iloc[i]})\n")


<div id="4-prueba-de-inferencia-modelo-predictivo" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>4. Prueba de Inferencia: Modelo Predictivo</h2> </div>

Se selecciona una muestra aleatoria de 5 ventanas temporales del conjunto de test del dataset predictivo. Se aplica el preprocesador, se genera el vector de probabilidades de degradación y se aplica el umbral de decisión optimizado (**0.55**) para obtener la clasificación binaria final (`maintain` / `downgrade_needed`). Se comparan las predicciones con las etiquetas reales.

In [ ]:
df_p = pd.read_csv('data/processed/dataset_predictivo.csv')
sample_p = df_p[df_p['split'] == 'test'].sample(5, random_state=42)
X_p = sample_p[meta['predictive_features']]
y_true_p = sample_p['downgrade_needed']

threshold = meta['selected_threshold']

# Pipeline de inferencia
X_p_scaled = prep_p.transform(X_p)
probs_p = mod_p.predict_proba(X_p_scaled)[:, 1]
preds_p = (probs_p >= threshold).astype(int)

print("--- INFERENCIA PREDICTIVA ---")
for i, (idx, row) in enumerate(X_p.iterrows()):
    real_label = "downgrade_needed (1)" if y_true_p.iloc[i] == 1 else "maintain (0)"
    pred_label = "downgrade_needed (1)" if preds_p[i] == 1 else "maintain (0)"
    
    print(f"Muestra {i+1} | Input P10: {row['throughput_p10']:.2f} Mbps, Tendencia: {row['throughput_slope']:.4f}, Perfil Actual: {row['current_profile']}")
    print(f"           | Probabilidad de Degradación: {probs_p[i]:.4f} (Umbral: {threshold})")
    print(f"           | Predicción: {pred_label} (Real: {real_label})\n")


<div id="5-conclusion" style="color:#37475a; border-bottom: 7px solid orange; width: 100%; margin-bottom: 15px; padding-bottom: 2px"><h2>5. Conclusión</h2> </div>

Los resultados de inferencia confirman que el pipeline de predicción es consistente y determinista: las mismas entradas producen siempre las mismas salidas, sin modificar los pesos internos del modelo. Los artefactos de la Fase 1 se encuentran correctamente empaquetados y verificados, listos para ser integrados en la arquitectura de ejecución en tiempo real que se diseñará en la **Fase 2**.